# 05 — Convert the Wikipedia-backed sources (Kaggle, CPU)

Converts **HoVer, FEVER and Natural Questions** to unified records: `<name>.jsonl` + `<name>.stats.json` in `/kaggle/working`, the notebook's output. Their evidence lives in Wikipedia dumps; training needs only the converted records.

**FEVEROUS is postponed:** its page database needs ~64 GB of disk at the peak, and a Kaggle session has ~55 GB.

- **Settings:** Accelerator → None (CPU), Internet → On.
- **Disk:** ~5 GB at the peak. Dumps go to `/tmp/raw` (not saved); NQ parquet files are deleted as soon as they are read.
- **Smoke run first:** run all cells interactively with `LIMIT = 200` and check the samples. Then set `LIMIT = None` and use **Save Version → Save & Run All (Commit)**: the full run (~1 h) continues in the background.
- **Hand-off** (on the laptop, from the repo root): `kaggle kernels output <user>/<notebook-slug> -p data/public`, then rebuild the mixes (last cell).

In [ ]:
# Environment: code from GitHub at a fixed commit, kept outside /kaggle/working so it is not saved as output.
COMMIT = "main"   # set a commit hash for a reproducible run

!rm -rf /tmp/JEV_Judge && git clone -q https://github.com/Nefelim584/JEV_Judge.git /tmp/JEV_Judge
!cd /tmp/JEV_Judge && git checkout -q {COMMIT} && git log -1 --format='%h %ad %s' --date=short
!pip install -q --no-deps --ignore-requires-python -e /tmp/JEV_Judge
!pip install -q loguru
!python --version
!df -h /kaggle/working /tmp | tail -2

In [ ]:
# Parameters
LIMIT = 200                       # smoke run; None = full conversion
DATASETS = ["hover", "fever", "nq"]
RAW = "/tmp/raw"                  # dumps, not saved
OUT = "/kaggle/working"           # notebook output: only the converted files

## Convert

One dataset per call, so a failure in one does not stop the others. Approximate full-run times: HoVer ~5 min, FEVER ~20 min (one pass over the 7.6 GB dump inside its zip), NQ ~30–40 min (37 parquet files).

In [ ]:
import os, subprocess, sys

env = {**os.environ, "JEVLITE_DROP_RAW": "1"}
for name in DATASETS:
    cmd = [sys.executable, "/tmp/JEV_Judge/scripts/convert_public.py", "--datasets", name, "--out", OUT, "--raw-dir", RAW]
    if LIMIT is not None:
        cmd += ["--limit", str(LIMIT)]
    print("$", " ".join(cmd), flush=True)
    subprocess.run(cmd, check=False, cwd="/tmp/JEV_Judge", env=env)
    !df -h /tmp | tail -1

## Check

Counts per split, type and label, then three random records per dataset. Look for: clean evidence text (no `-LRB-`, no `` `` `` quotes), states that read like retrieved chunks, plausible labels.

In [ ]:
import json, random
from collections import Counter
from pathlib import Path

for name in DATASETS:
    path = Path(OUT) / f"{name}.jsonl"
    if not path.exists():
        print(f"== {name}: MISSING"); continue
    records = [json.loads(line) for line in path.open()]
    print(f"== {name}: {len(records)} records")
    print("  ", Counter((r["split"], r["type"], r["target"]) for r in records).most_common(12))
    for r in random.Random(0).sample(records, min(3, len(records))):
        state = r["state"] if isinstance(r["state"], str) else " ||| ".join(r["state"])
        print(f"  [{r['type']} → {r['target']}] {r['prompt']}")
        print("   ", state[:600].replace("\n", " / "))
    print()

## Hand-off (on the laptop)

```bash
kaggle kernels output <user>/<notebook-slug> -p data/public      # hover, fever, nq (.jsonl + .stats.json)
uv run python scripts/build_mix.py --config configs/mixes/stage_a.yaml --out data/mix/stage_a --epochs 2
uv run python scripts/build_mix.py --config configs/mixes/stage_a_nosa.yaml --out data/mix/stage_a_nosa --epochs 2
```

Then publish the mixes as a new version of the `jevlite-mix` dataset: `kaggle datasets version -p data/mix --dir-mode zip -m "stage_a + FEVER, HoVer, NQ"`.